# 第 7 周练习 — 本地 GPU 上的 QLoRA（「价格合适」放大版）

## 练习目标

第 6 周在 **CPU** 上微调了一个微型模型。本练习把同样的定价任务接到 **QLoRA（4bit 量化 + LoRA）**：

- 以 4bit 加载 **Qwen2.5-1.5B-Instruct**
- 在约 **8GB RTX 2070** 上用定价数据做 QLoRA 微调
- 用同一批样本对比：**基础 4bit 模型** vs **微调后**，并对照第 6 周 / GPT 参考分

## 和本课 Week 7 的关系

| 概念 | 本练习里你会看到 |
|------|------------------|
| 4bit 量化（NF4） | `BitsAndBytesConfig(load_in_4bit=True, ...)` |
| QLoRA | `prepare_model_for_kbit_training` + `LoraConfig` / `get_peft_model` |
| 只训练答案 | label 里 prompt 段填 `-100`（忽略损失） |
| 适配器开关对比 | `model.disable_adapter()` 测 base |

## 怎么跑

1. 需要 venv 里已装好 **CUDA torch + bitsandbytes**
2. 注意：`uv run` 可能把环境重新同步成 CPU torch — 请直接用该 venv 的 Python 启动 notebook
3. 数据复用作者第 6 周的 `data_small.pkl`（路径见代码常量 `DATA`）


In [1]:
# ========== 导入、定位仓库、加载第 6 周定价数据 ==========
# 标准库：sys（改模块搜索路径）、pickle（读 .pkl）、re（后面抽价格）、random（打乱训练）
import sys, pickle, re, random
# Path：跨平台路径定位
from pathlib import Path
# PyTorch 张量与 CUDA
import torch
# transformers：因果 LM、分词器、4bit 量化配置
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# peft：LoRA 配置、挂载适配器、kbit 训练准备
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 从当前目录向上找含 week6/pricer/items.py 的仓库根
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week6/pricer/items.py").exists())
# 把 week6 加进 sys.path，才能 import pricer.items
sys.path.insert(0, str(REPO / "week6"))
# Item：课程定价样本的 Pydantic/数据模型
from pricer.items import Item  # noqa: E402
# 复用第 6 周策展的小数据集 pickle（路径字符串保持原样）
DATA = REPO / "community-contributions/NicholasDean/week6/data_small.pkl"  # reuse week 6 curation

# 二进制读入 pickle blob
blob = pickle.load(open(DATA, "rb"))
# 训练集：把每条 dict 校验成 Item
train = [Item.model_validate(d) for d in blob["train"]]
# 评估子集：测试集前 50 条
SAMPLE = [Item.model_validate(d) for d in blob["test"][:50]]
# 打印 GPU 名与训练条数，确认环境就绪
print("cuda:", torch.cuda.get_device_name(0), "| train:", len(train))


cuda: NVIDIA GeForce RTX 2070 SUPER | train: 16801


## 以 4bit 加载 Qwen2.5-1.5B（QLoRA 底座）

**NF4** 4bit 量化大约把权重大小压到原来的约 1/4。这里计算 dtype 用 **float16**（Turing 系 GPU 通常没有稳定的 bf16 路径）。


In [2]:
# ========== 4bit 加载底座 + 挂上 LoRA 适配器 ==========
# Hugging Face 模型 id（勿改字符串）
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# 4bit NF4 + float16 计算
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
# 加载分词器
tok = AutoTokenizer.from_pretrained(MODEL)
# 若无 pad_token，则退回 eos_token
tok.pad_token = tok.pad_token or tok.eos_token
# 左侧截断：尽量保留序列末尾（价格目标所在位置）
tok.truncation_side = "left"                          # keep the END (the price target)
# 量化加载到 CUDA
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="cuda")
# 为 kbit 训练做准备（梯度/层冻结等）
model = prepare_model_for_kbit_training(model)
# 注入 LoRA：r=16, alpha=32；只改注意力与 MLP 投影层
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
# 打印可训练参数占比
model.print_trainable_parameters()
# 看加载后已占用的显存（GB）
print("VRAM after load (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))


W0623 11:56:50.141000 35292 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
VRAM after load (GB): 1.75


## QLoRA 微调（在 GPU 上手写训练循环）

下面不用 Trainer：自己拼 `input_ids` / `labels`，用 AdamW 跑 3 个 epoch。重点是 **prompt 段 label=-100**，只对价格答案算损失。


In [3]:
# ========== 构造样本 + 手写 3-epoch 训练循环 ==========
# 另起别名 _t，方便在循环里写短一些（与原代码一致）
import torch as _t

def example(it):
    # 提示以「Price is $」一类结尾；答案是价格 -> 把 prompt 段 label 置 -100，只训练答案
    # 对测试提示分词并截断到 210
    p = tok(it.test_prompt(), truncation=True, max_length=210)["input_ids"]
    # 答案：四舍五入到整数美元 + ".00" + eos
    a = tok(f"{round(it.price)}.00" + tok.eos_token, add_special_tokens=False)["input_ids"]
    # 拼接 token；labels 前半全 -100（忽略），后半是答案 id
    return p + a, [-100] * len(p) + a

# 取训练集前 1200 条做成 (ids, labels) 列表
data = [example(it) for it in train[:1200]]
# 只优化 requires_grad=True 的参数（LoRA），学习率 2e-4
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
# 切换到训练模式
model.train()
# 3 个 epoch
for epoch in range(3):
    # 每轮用固定种子打乱，保证可复现打乱顺序
    random.Random(epoch).shuffle(data)
    # 逐条（batch size=1）前向 + 反传
    for ids, labels in data:
        # input_ids 放到 CUDA，形状 [1, seq]
        ids_t = _t.tensor([ids], device="cuda")
        # 前向：传入 labels 以计算交叉熵 loss
        out = model(input_ids=ids_t, labels=_t.tensor([labels], device="cuda"))
        # 反传
        out.loss.backward()
        # 更新参数并清空梯度
        opt.step(); opt.zero_grad()
    # 每个 epoch 结束打印进度
    print(f"epoch {epoch + 1} done")
# 峰值显存（GB）
print("peak VRAM (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


c:\Users\Nicholas Dean\projects\llm_engineering\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


epoch 1 done


epoch 2 done


epoch 3 done
peak VRAM (GB): 2.49


## 比较：基础 Qwen（关适配器）vs QLoRA 微调

同一套 `evaluate`：先测微调模型，再用 `disable_adapter()` 关掉 LoRA 测 4bit 底座，输出 MAE 与 hit-rate。


In [4]:
# ========== 推理抽价格 + 在 SAMPLE 上对比 base / QLoRA ==========
def get_price(text):
    # 从生成文本里用正则抽出第一个数字（去掉千分位逗号）
    nums = re.findall(r"[-+]?\d*\.?\d+", text.replace(",", ""))
    # 有数字则转 float，否则 0.0
    return float(nums[0]) if nums else 0.0

def local_price(item):
    # 对商品测试提示分词并放到 GPU，截断到 220
    enc = tok(item.test_prompt(), return_tensors="pt", truncation=True, max_length=220).to("cuda")
    # 推理关闭梯度
    with torch.no_grad():
        # 贪婪解码最多 8 个新 token
        out = model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    # 只解码「新生成」那一段，再解析成价格
    return get_price(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))

def evaluate(items):
    # 绝对误差列表
    errs = [abs(local_price(it) - it.price) for it in items]
    # hit：误差 ≤ 20% 真实价 或 ≤ 40 美元
    hits = sum(e <= 0.2 * it.price or e <= 40 for e, it in zip(errs, items))
    # 返回平均绝对误差 MAE 与 hit-rate
    return sum(errs) / len(errs), hits / len(items)

# 评估模式
model.eval()
# 先评估带 LoRA 的微调模型
mae_ft, hit_ft = evaluate(SAMPLE)
# 临时关掉适配器，评估纯 4bit 底座
with model.disable_adapter():
    mae_base, hit_base = evaluate(SAMPLE)
# 打印对比结果（文案保持英文，便于和参考分对齐）
print(f"base Qwen2.5-1.5B (4-bit):   MAE=${mae_base:,.2f}  hit-rate={hit_base:.0%}")
print(f"QLoRA fine-tuned Qwen-1.5B:   MAE=${mae_ft:,.2f}  hit-rate={hit_ft:.0%}")
print("references -> week6 distilgpt2 LoRA: $52.78 (76%) | gpt-4o-mini zero-shot: $22.80 (86%)")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


c:\Users\Nicholas Dean\projects\llm_engineering\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


base Qwen2.5-1.5B (4-bit):   MAE=$47.82  hit-rate=70%
QLoRA fine-tuned Qwen-1.5B:   MAE=$42.71  hit-rate=80%
references -> week6 distilgpt2 LoRA: $52.78 (76%) | gpt-4o-mini zero-shot: $22.80 (86%)
